# Memory Operations

**Module:** 13 — AI Memory

Capture, retrieve, consolidate, forget — pipelines and metrics.

---


## How to Use This Notebook

1. Read each major section fully before running code — the narrative carries the design intuition.
2. Run code cells top-to-bottom. They are self-contained demos (stdlib / light mocks) unless noted.
3. Treat API keys as **placeholders**: set `OPENAI_API_KEY`, `ANTHROPIC_API_KEY`, etc. in your environment — never hardcode secrets.
4. Complete **Try it yourself** exercises; they are the difference between recognition and skill.


## Learning Objectives

By the end of this notebook, you will be able to:

- Implement core CRUD-like memory operations with dedupe
- Build a retrieve → rerank → pack pipeline
- Apply consolidation and forgetting strategies
- Define observability metrics for memory quality


## Core Operations

### Definition
The operational verbs of a memory system: create/upsert, read/retrieve, update, delete/forget, and list/export.

### Why it matters
Products fail when write paths are ad hoc. Explicit operations enable testing, authz, and compliance.

### How it works
Expose a small API used by the agent runtime. Every write should validate tenant, type, size limits, and PII policy.

### Intuition
Treat memory like a microservice with a boring, strict API — not a junk drawer.

### Pitfalls
- Silent failures on write
- Duplicates from non-idempotent creates
- Delete that only soft-hides in one index but not others

### When to use
Any production memory layer — standardize these ops first.


In [ ]:
# Demo 1: idempotent upsert + delete
from dataclasses import dataclass
from typing import Optional
import hashlib

@dataclass
class Record:
    id: str
    tenant_id: str
    text: str

class MemoryOps:
    def __init__(self):
        self.store: dict[str, Record] = {}

    def _id(self, tenant_id: str, text: str) -> str:
        return hashlib.sha256(f"{tenant_id}\0{text}".encode()).hexdigest()[:12]

    def upsert(self, tenant_id: str, text: str) -> str:
        rid = self._id(tenant_id, text)
        self.store[rid] = Record(rid, tenant_id, text)
        return rid

    def get(self, rid: str) -> Optional[Record]:
        return self.store.get(rid)

    def forget(self, rid: str) -> bool:
        return self.store.pop(rid, None) is not None

ops = MemoryOps()
a = ops.upsert("t1", "likes cats")
b = ops.upsert("t1", "likes cats")  # same id
print(a, b, a == b, len(ops.store))
print("forgot", ops.forget(a), "size", len(ops.store))


## Retrieval Pipeline

### Definition
A multi-stage path from user query to budgeted memory snippets: filter → ANN/keyword recall → rerank → pack.

### Why it matters
Raw top-k from a vector DB is rarely optimal. Pipelines improve precision and control cost.

### How it works
1. **Authorize & scope** tenant/user  
2. **Recall** candidates (vector, BM25, SQL)  
3. **Rerank** (cross-encoder, LLM, or heuristics: importance, recency)  
4. **Pack** under token budget with diversity  
5. **Log** what was injected for eval

### Intuition
Cast a wide net, then carefully choose what goes on the prompt plate.

### Pitfalls
- Skipping filters
- Reranking with an expensive model on 1000 candidates
- No diversity → 5 near-duplicate memories

### When to use
Default path for every retrieve-then-generate agent.


```mermaid
flowchart TD
  Q[Query + user/tenant] --> F[Metadata filters]
  F --> C[Candidate recall]
  C --> R[Rerank]
  R --> P[Pack to budget]
  P --> L[LLM]
  L --> W[Optional writeback]
```


In [ ]:
# Demo 2: retrieve → rerank → pack
from datetime import datetime, timezone

candidates = [
    {"text": "prefers UTC", "importance": 0.9, "ts": "2026-07-01T00:00:00+00:00", "overlap": 1},
    {"text": "prefers UTC timestamps in reports", "importance": 0.8, "ts": "2026-08-01T00:00:00+00:00", "overlap": 2},
    {"text": "likes pizza", "importance": 0.2, "ts": "2026-08-01T00:00:00+00:00", "overlap": 0},
    {"text": "project uses Qdrant", "importance": 0.7, "ts": "2026-06-01T00:00:00+00:00", "overlap": 0},
]

def rerank(cands, now=None):
    now = now or datetime(2026, 8, 1, tzinfo=timezone.utc)
    scored = []
    for c in cands:
        ts = datetime.fromisoformat(c["ts"])
        age_days = max((now - ts).days, 0)
        recency = 1.0 / (1.0 + age_days / 30.0)
        score = 0.5 * c["overlap"] + 0.3 * c["importance"] + 0.2 * recency
        scored.append((score, c))
    scored.sort(reverse=True, key=lambda x: x[0])
    return scored

def pack(scored, budget=60):
    out, used = [], 0
    for score, c in scored:
        if c["overlap"] == 0 and score < 0.4:
            continue
        if used + len(c["text"]) > budget:
            break
        out.append((round(score, 3), c["text"]))
        used += len(c["text"])
    return out

print(pack(rerank(candidates)))


In [ ]:
# Demo 3: simple MMR-style diversity (avoid near-duplicates)
def jaccard(a: str, b: str) -> float:
    aa, bb = set(a.lower().split()), set(b.lower().split())
    return len(aa & bb) / len(aa | bb) if aa | bb else 0.0

def mmr_select(texts: list[str], k: int = 3, lambda_=0.7):
    selected = []
    remaining = texts[:]
    while remaining and len(selected) < k:
        def score(t):
            rel = len(t.split())  # fake relevance
            div = max((jaccard(t, s) for s in selected), default=0.0)
            return lambda_ * rel - (1 - lambda_) * div * 10
        best = max(remaining, key=score)
        selected.append(best)
        remaining.remove(best)
    return selected

texts = [
    "prefers UTC",
    "prefers UTC timestamps",
    "uses Qdrant vector database",
    "qdrant for vectors",
    "timezone America/New_York",
]
print(mmr_select(texts, k=3))


### Try it yourself — Pipeline

1. Add a hard filter: drop candidates with importance < 0.3 unless overlap >= 2.
2. Log a JSON line per retrieval: query, ids, scores, packed texts.


## Consolidation & Forgetting

### Definition
Consolidation merges/summarizes many episodes into fewer semantic facts. Forgetting removes or decays low-value or illegal-to-keep data.

### Why it matters
Unbounded memory degrades retrieval and increases risk. Healthy systems compress and prune.

### How it works
Batch jobs: cluster similar episodes → LLM summarize → upsert semantic fact → archive/delete raw. Apply TTL and user deletes.

### Intuition
Sleep consolidates memories; your cron job should too.

### Pitfalls
- Summaries that invent facts not in sources
- Forgetting without cascading to all indexes
- No user-facing 'what do you know about me?' export

### When to use
Long-running agents, privacy regimes, cost control at scale.


In [ ]:
# Demo 4: consolidate near-duplicate strings into one fact
from collections import defaultdict

raw = [
    "User likes dark mode",
    "user prefers dark mode",
    "dark mode preference",
    "Uses PostgreSQL",
]

def normalize(s: str) -> str:
    return " ".join(sorted(set(s.lower().replace("user", "").split())))

groups = defaultdict(list)
for r in raw:
    groups[normalize(r)].append(r)

consolidated = {k: max(v, key=len) for k, v in groups.items()}  # keep longest
print(consolidated)


In [ ]:
# Demo 5: importance decay forgetting
def decay_importance(importance: float, age_days: float, half_life=45.0) -> float:
    return importance * (0.5 ** (age_days / half_life))

for age in [0, 30, 90, 180]:
    print(age, round(decay_importance(0.9, age), 3))

def should_forget(importance, age_days, floor=0.15) -> bool:
    return decay_importance(importance, age_days) < floor

print("forget 180d?", should_forget(0.9, 180))


## Observability Metrics

| Metric | Intent |
|--------|--------|
| `memory_recall_at_k` | Eval quality vs labeled set |
| `inject_rate` | Are we actually using memory? |
| `avg_injected_tokens` | Cost/budget health |
| `write_dedupe_ratio` | Idempotency working? |
| `stale_hit_rate` | Too many old facts? |
| `cross_tenant_hits` | Must stay 0 |
| `forget_lag_seconds` | GDPR delete completeness |

### Pipeline walkthrough (support agent)
1. User: "Did we already try resetting the cache?"  
2. Retrieve episodes about cache reset for this ticket  
3. Rerank by ticket_id + recency  
4. Inject top 2  
5. Answer with citation to episode ids  
6. Write new episode: "Advised cache reset again; user declined"


### Try it yourself — Metrics

1. Implement a toy `recall_at_k(predicted_ids, relevant_ids, k)`.
2. Simulate 100 retrievals and compute inject_rate if packing returns non-empty.

**Stretch:** Build a consolidation job that refuses to invent tokens not present in sources (token subset check).


## Glossary / Key Terms

| Term | Meaning |
|------|---------|
| `rerank` | Reordering candidates with a stronger model/heuristic |
| `MMR` | Maximal Marginal Relevance — relevance with diversity |
| `consolidation` | Compressing many memories into fewer facts |
| `decay` | Reducing importance with age |
| `writeback` | Persisting new memories after a turn |


## End-to-End Retrieval SLA Budget

| Stage | p95 budget |
|-------|------------|
| Auth + filters | 5 ms |
| Embed query (or cache) | 40 ms |
| ANN search | 30 ms |
| Rerank top 20 | 50 ms |
| Pack | 2 ms |
| **Total retrieve** | **~127 ms** |

If voice agents need <300 ms to first token, memory must hit cache or run in parallel with other prep.


In [ ]:
# Eval: recall@k and MRR
def recall_at_k(predicted, relevant, k):
    pred = predicted[:k]
    return len(set(pred) & set(relevant)) / len(relevant) if relevant else 0.0

def mrr(predicted, relevant):
    for i, p in enumerate(predicted, 1):
        if p in relevant:
            return 1.0 / i
    return 0.0

pred = ["a", "x", "b"]
rel = ["b", "c"]
print(recall_at_k(pred, rel, 3), mrr(pred, rel))


In [ ]:
# GDPR-ish cascade delete
indexes = {
    "postgres": {"m1": "likes cats", "m2": "tz=UTC"},
    "vector": {"m1": [0.1], "m2": [0.2]},
    "cache": {"m1": "likes cats"},
}

def cascade_forget(mem_id: str):
    report = {}
    for name, store in indexes.items():
        report[name] = store.pop(mem_id, None) is not None
    return report

print(cascade_forget("m1"))
print(indexes)


### Try it yourself — Ops deepen

1. Implement diversity penalty in packer using Jaccard > 0.8 skip.
2. Create a nightly consolidation job stub with dry-run mode.


## Key Takeaways

- Standardize upsert/get/forget before adding fancy ANN
- Retrieval is a pipeline, not a single search call
- Consolidate and forget or quality will rot
- Instrument memory like any production dependency
